# M1-INF4 — Python Workshop for Data Analysis
## Session 5 — Data Integration Case Study

### A retail decision supported by reliable data
You are a small analytics team working for **Campus Tech**, a fictional retailer of computer accessories and equipment. The manager wants to compare stores before deciding where to investigate further.

> “The store with the highest total sales is our best-performing store. We should copy its approach everywhere.”

**Your mission:** integrate three sources, assess this claim and deliver an evidence-based recommendation.

This is a synthesis project. Use the pandas and Matplotlib techniques from Sessions 1–4. No new library or advanced modelling is required.

**All data are synthetic. Some records deliberately contain quality issues for training.** Findings describe this supplied sample, not a real company.

### What the data represent
The intended scope is **1–28 April 2025 inclusive**. This is an extract of completed orders, **not a complete record of all activity**. Coverage can differ between stores. A day absent from the extract is not evidence of zero sales or store closure.

| File | One row represents | Expected key | Purpose |
|---|---|---|---|
| `case_sales.csv` | One completed order for one product, possibly several units | `order_id` | Recorded quantity and unit selling price |
| `case_stores.csv` | One store | `store_id` | Store name, city and format |
| `case_products.csv` | One product | `product_id` | Product name and category |

Each order concerns **one product only**, so one valid order row is one order, not a line in a multi-product basket. The price is the final selling price per unit in EUR, after discounts and excluding VAT. Returns and cancellations are outside this extract.

Revenue per order = quantity × unit selling price. **Revenue is not profit**: costs, staffing, opening hours and customer traffic are unavailable.

The reference files should have unique keys, but this must be checked. Repeated store or product IDs in the sales file are normal.

### Data dictionary
| Field | Meaning and rule |
|---|---|
| `order_id` | Order identifier, required and unique after removing exact repeated records |
| `date` | Order date, expected format `YYYY-MM-DD`, within the study window |
| `store_id`, `product_id` | Reference keys; trim spaces and normalise case before joining |
| `quantity` | Positive whole number of units |
| `unit_price_eur` | Positive numeric unit price; a decimal comma may occur |
| `store_name`, `city`, `store_format` | Store attributes; “Small”/“Large” describes format, not observed floor area |
| `product_name`, `category` | Product attributes |

**Quality policy:** keep raw objects unchanged. Remove exact duplicate rows with an audit count. Convert invalid dates/numbers to missing values. Keep invalid or unmatched records in review tables, without inventing replacements. Do not remove a large but valid order simply because it is unusual. If duplicate reference keys disagree, investigate rather than arbitrarily keeping the first row.

### Files and group deliverable
Place this notebook in `notebooks/` and the three CSV files in the neighbouring `data/` folder. You may also put the CSV files beside the notebook.

Submit one completed notebook per group, the exported summary tables and three labelled figures. Record group members below. Divide the work, then review the entire analysis together: **every member must be able to explain the data unit, cleaning choices, joins and conclusions**.

Group members: *complete here.*

The sections below form one continuous analysis. Each checkpoint is a review of your own results, not a separate dataset.

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path('../data'), Path('data'), Path('.')]
DATA_DIR = next((p for p in candidates if all((p / f).exists() for f in
    ['case_sales.csv', 'case_stores.csv', 'case_products.csv'])), None)
if DATA_DIR is None:
    raise FileNotFoundError('Place the three case_*.csv files in data/ or beside this notebook.')
OUT_DIR = Path('case_study_outputs')
OUT_DIR.mkdir(exist_ok=True)

sales_raw = pd.read_csv(DATA_DIR / 'case_sales.csv', dtype='string')
stores_raw = pd.read_csv(DATA_DIR / 'case_stores.csv', dtype='string')
products_raw = pd.read_csv(DATA_DIR / 'case_products.csv', dtype='string')
for name, frame in [('Sales', sales_raw), ('Stores', stores_raw), ('Products', products_raw)]:
    print(name, frame.shape)
    display(frame.head())

Sales (252, 6)


,order_id,date,store_id,product_id,quantity,unit_price_eur
0,O0001,2025-04-21,s1,P02,3,19.90
1,O0002,2025-04-14,S1,p02,2,39.90
2,O0003,2025-04-14,S1,P04,1,"179,00"
3,O0004,2025-04-31,S1,P03,1,179.00
4,O0005,2025-04-09,S1,P02,two,19.90


Stores (4, 4)


,store_id,store_name,city,store_format
0,S1,Paris Centre,Paris,Small
1,S2,Paris Est,Paris,Large
2,S3,Lyon Centre,Lyon,Small
3,S4,Lille Centre,Lille,Large


Products (7, 3)


,product_id,product_name,category
0,P01,Keyboard,Accessories
1,P02,Mouse,Accessories
2,P03,Monitor,Screens
3,P04,Headset,Audio
4,P05,Webcam,Video


## 1. Understand and audit the sources
1. Report row and column counts, missing values and exact duplicate counts for each source.
2. Check the expected keys. Explain why repeated `store_id` values are acceptable in sales but potentially dangerous in stores.
3. Write the observation unit and the proposed join keys before joining anything.

**Checkpoint:** your audit should distinguish missing data, repeated records and normal repeated foreign keys.

In [6]:
sources = {
    "Sales": sales_raw,
    "Stores": stores_raw,
    "Products": products_raw
}

expected_key = {
    "Sales": "order_id",
    "Stores": "store_id",
    "Products": "product_id"
}

audit_rows = []

for name, df in sources.items():
    key = expected_key[name]

    # Count repeated non-missing key values
    repeated_keys = int(
        df.loc[df[key].notna(), key].duplicated().sum()
    )

    # Repeated primary keys are dangerous
    dangerous_key_repeated = (
        repeated_keys if name in ["Stores", "Products"] else 0
    )

    audit_rows.append({
        "source": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "key": key,
        "key_missing": int(df[key].isna().sum()),
        "normal_key_repeated": repeated_keys,
        "dangerous_key_repeated": dangerous_key_repeated
    })

audit = pd.DataFrame(audit_rows).set_index("source")
display(audit)

print(
    "Repeated store_id values are acceptable in Sales because "
    "store_id is a foreign key there. They are potentially dangerous "
    "in Stores because store_id should be a primary key."
)

print(
    "Observation units:\n"
    "- Sales: each record represents a sale of a product at a store "
    "on a given date.\n"
    "- Stores: each record represents one store.\n"
    "- Products: each record represents one product."
)

print(
    "Proposed join keys:\n"
    "- Sales to Stores: store_id.\n"
    "- Sales to Products: product_id."
)

,rows,columns,missing_cells,exact_duplicate_rows,key,key_missing,normal_key_repeated,dangerous_key_repeated
source,,,,,,,,
Sales,252,6,1,2,order_id,0,2,0
Stores,4,4,0,0,store_id,0,0,0
Products,7,3,0,1,product_id,0,1,1


Repeated store_id values are acceptable in Sales because store_id is a foreign key there. They are potentially dangerous in Stores because store_id should be a primary key.
Observation units:
- Sales: each record represents a sale of a product at a store on a given date.
- Stores: each record represents one store.
- Products: each record represents one product.
Proposed join keys:
- Sales to Stores: store_id.
- Sales to Products: product_id.


**Your interpretation**

One row in `case_sales.csv` is one order for one product. One row in `case_stores.csv` is one store, one row in `case_products.csv` is one product. `store_id` links sales to stores, `product_id` links sales to products.

Repeated `store_id` in sales is normal, a store naturally has many orders. A repeated `store_id` in `stores` itself would be dangerous, since it's supposed to identify exactly one store there. If it repeated, joining could match one order to two store rows and duplicate it. `stores` is clean, but `products` has one exact duplicate (`P01`), harmless since it's identical, but still has to be removed before joining or it would multiply matches.

## 2. Prepare reliable tables
Create working copies. Apply the quality policy above.

1. Count and remove **exact** duplicate records in all three tables.
2. Normalise identifiers, parse dates and convert the two numeric fields. Handle decimal commas.
3. Flag missing IDs, failed conversions, non-positive prices, quantities that are not positive integers, and dates outside the study window.
4. Produce `sales_valid` and `sales_review`. Keep the rejected records and the reason flags.
5. Validate unique, non-missing reference keys and unique valid order IDs.

**Checkpoint:** input rows = exact duplicates removed + invalid rows retained for review + valid rows. Multiple reason flags can describe the same rejected row, so do not sum flags as if they were distinct rows.

In [11]:
# 1
sales = sales_raw.copy()
stores = stores_raw.copy()
products = products_raw.copy()

input_rows = len(sales)   # point de départ de la réconciliation

# Comptage avant la suppression
dup_counts = {
    "sales": int(sales.duplicated().sum()),
    "stores": int(stores.duplicated().sum()),
    "products": int(products.duplicated().sum()),
}
print(dup_counts)

# Suppression
sales = sales.drop_duplicates().reset_index(drop=True)
stores = stores.drop_duplicates().reset_index(drop=True)
products = products.drop_duplicates().reset_index(drop=True)

#2 :

for df, cols in [(sales, ["order_id", "store_id", "product_id"]),
                 (stores, ["store_id"]),
                 (products, ["product_id"])]:
    for c in cols:
        df[c] = df[c].str.strip().str.upper()

# Date
sales["date_parsed"] = pd.to_datetime(
    sales["date"].str.strip(), format="%Y-%m-%d", errors="coerce"
)

# Quantité
sales["quantity_num"] = pd.to_numeric(sales["quantity"].str.strip(), errors="coerce")

# Prix
sales["unit_price_num"] = pd.to_numeric(
    sales["unit_price_eur"].str.strip().str.replace(",", ".", regex=False),
    errors="coerce",
)

# 3. flag issues (one boolean column per reason, a row can trigger more than one)
study_start = pd.Timestamp("2025-04-01")
study_end = pd.Timestamp("2025-04-28")
reason_missing_id = sales["order_id"].isna() | sales["store_id"].isna() | sales["product_id"].isna()
reason_invalid_date = sales["date_parsed"].isna() | (sales["date_parsed"] < study_start) | (sales["date_parsed"] > study_end)
reason_invalid_price = sales["unit_price_num"].isna() | (sales["unit_price_num"] <= 0)
reason_invalid_quantity = (
    sales["quantity_num"].isna()
    | (sales["quantity_num"] <= 0)
    | (sales["quantity_num"] % 1 != 0)
)
invalid_mask = reason_missing_id | reason_invalid_date | reason_invalid_price | reason_invalid_quantity

# 4. sales_valid and sales_review, keeping the reason flags
sales_review = sales.loc[invalid_mask, ["order_id", "date", "store_id", "product_id", "quantity", "unit_price_eur"]].copy()
sales_review["reason_missing_id"] = reason_missing_id[invalid_mask]
sales_review["reason_invalid_date"] = reason_invalid_date[invalid_mask]
sales_review["reason_invalid_price"] = reason_invalid_price[invalid_mask]
sales_review["reason_invalid_quantity"] = reason_invalid_quantity[invalid_mask]

sales_valid = sales.loc[~invalid_mask, ["order_id", "store_id", "product_id"]].copy()
sales_valid["date"] = sales.loc[~invalid_mask, "date_parsed"]
sales_valid["quantity"] = sales.loc[~invalid_mask, "quantity_num"].astype(int)
sales_valid["unit_price_eur"] = sales.loc[~invalid_mask, "unit_price_num"]

print()
print("sales_valid rows:", len(sales_valid))
print("sales_review rows:", len(sales_review))
display(sales_review[["order_id", "reason_missing_id", "reason_invalid_date", "reason_invalid_price", "reason_invalid_quantity"]])

# row accounting: do not sum the 4 flag columns, a row can carry more than one flag
assert len(sales) == len(sales_valid) + len(sales_review)

# 5. validate reference keys and order IDs
stores_clean = stores
products_clean = products
duplicate_rows_removed = dup_counts["sales"]

assert stores_clean["store_id"].notna().all() and stores_clean["store_id"].is_unique
assert products_clean["product_id"].notna().all() and products_clean["product_id"].is_unique
assert sales_valid["order_id"].notna().all() and sales_valid["order_id"].is_unique
print("Reference keys and valid order IDs are unique and non-missing.")

{'sales': 2, 'stores': 0, 'products': 1}

sales_valid rows: 243
sales_review rows: 7


,order_id,reason_missing_id,reason_invalid_date,reason_invalid_price,reason_invalid_quantity
3,O0004,False,True,False,False
4,O0005,False,False,False,True
5,O0006,False,False,True,False
6,O0007,False,False,False,True
7,O0008,False,False,True,False
8,O0009,False,False,False,True
11,O0012,False,True,False,False


Reference keys and valid order IDs are unique and non-missing.


**Your interpretation**

Which records did you exclude, repair or keep? Why?

*Write your answer here.*

## 3. Integrate and reconcile
1. Join `sales_valid` to stores with a **left join**, `validate="many_to_one"` and an indicator. Inspect unmatched keys.
2. Join that result to products with the same controls. Use a different indicator name, or rename/remove the first indicator before the second join.
3. Check that neither join changes the row count.
4. Keep orders missing either reference in `integration_review`. Create `analysis` using orders matched to **both** references.
5. Present a reconciliation table from raw input to the final analytical population.

**Checkpoint:** never interpret an unknown product as zero revenue, and never allow a duplicated reference key to multiply orders. State how many orders the analysis excludes.

In [ ]:
# Join the sources and audit coverage

# 1. left join to stores, with validation and an indicator
with_store = sales_valid.merge(
    stores_clean, on="store_id", how="left", validate="many_to_one", indicator="store_match"
)
assert len(with_store) == len(sales_valid)  # 3. row count unchanged after the store join
print("Store match:")
print(with_store["store_match"].value_counts())

# 2. left join to products, different indicator name
with_both = with_store.merge(
    products_clean, on="product_id", how="left", validate="many_to_one", indicator="product_match"
)
assert len(with_both) == len(with_store)  # 3. row count unchanged after the product join
print()
print("Product match:")
print(with_both["product_match"].value_counts())

# 4. orders missing either reference go to integration_review; the rest become analysis
unmatched = with_both["store_match"].eq("left_only") | with_both["product_match"].eq("left_only")
integration_review = with_both.loc[unmatched].copy()
analysis = with_both.loc[~unmatched].drop(columns=["store_match", "product_match"]).copy()

print()
print("Orders excluded for an unmatched store or product reference:", len(integration_review))
display(integration_review[["order_id", "store_id", "product_id", "store_match", "product_match"]])
print("analysis rows:", len(analysis))

# 5. reconciliation table, from raw input to the final analytical population
reconciliation = pd.DataFrame([
    {"stage": "Raw sales rows", "rows": len(sales_raw)},
    {"stage": "Exact duplicate rows removed", "rows": -duplicate_rows_removed},
    {"stage": "Invalid rows set aside (sales_review)", "rows": -len(sales_review)},
    {"stage": "Unmatched reference set aside (integration_review)", "rows": -len(integration_review)},
    {"stage": "Final analysis population", "rows": len(analysis)},
])
display(reconciliation)

assert len(sales_raw) == duplicate_rows_removed + len(sales_review) + len(integration_review) + len(analysis)

**Your interpretation**

A left join keeps every row from `sales_valid`, matched or not, so it's the only way to actually see which orders reference a store or product ID that doesn't exist (`S99`, `P99`). An inner join would have dropped those two orders silently, with no record of what was removed or why. Joining left first, then checking `store_match`/`product_match` before splitting into `analysis` and `integration_review`, keeps a clear trail of what got excluded and why.

## 4. Build indicators that answer the question
For each order in `analysis`, calculate `revenue_eur`.

Create `store_summary` with, for each store:
- order count and total units;
- total revenue (EUR);
- mean and median revenue per order (EUR).

Also create a revenue summary by product category and a daily revenue table for each store. Check that the totals reconcile with the order-level revenue.

Compare the store with the highest total revenue to the store with the highest mean revenue per order. Do they coincide? Explain the role of order counts and product mix.

**Important:** `quantity` counts units, not orders. Mean revenue per order = total revenue / number of orders. Round for presentation, not before aggregation. Avoid calling this a profitability or productivity ranking.

In [ ]:
# Compute order-level and aggregated indicators
# Write your code here.

**Your interpretation**

Does the same store lead for total and mean revenue? What can explain a difference?

*Write your answer here.*

## 5. Communicate with three complementary figures
Before coding each figure, write **the question**, **the observation unit** and **the aggregation**.

1. A bar chart comparing total revenue by store, with order counts available alongside it.
2. A boxplot of order revenue by store, showing centre, spread and atypical values. Display group sizes. Do not remove atypical values without evidence of error.
3. A line chart of daily revenue by store over the study window. Use chronological dates, one labelled colour per store and a common calendar. **Leave missing store-days as missing**, not zero, because this is an extract.

Use clear titles, units, readable labels and legends when needed. Add a two-sentence interpretation and a limitation for each figure. Explain why the three figures answer different questions.

In [ ]:
# Figure 1 — Total revenue by store
# Write your code here.

**Your interpretation**

State the question, unit and aggregation; then interpret the chart.

*Write your answer here.*

In [ ]:
# Figure 2 — Distribution of order revenue
# Write your code here.

**Your interpretation**

State the question and observation unit; describe spread and a limitation.

*Write your answer here.*

In [ ]:
# Figure 3 — Daily sampled revenue
# Write your code here.

**Your interpretation**

What does one point represent? How did you treat missing dates?

*Write your answer here.*

## 6. Write a decision brief
Write **six to eight sentences** addressed to the manager. Include:
- a clear answer to the original claim;
- at least two numerical results, with units and population;
- a distinction between total revenue and mean or median order revenue;
- a data-quality or integration limitation and a sample-coverage limitation;
- one plausible explanation labelled as a hypothesis;
- one proportionate next action and the extra data needed.

Avoid “Store X is best” without defining best. Do not claim causality, profitability or complete activity from this extract.

**Your interpretation**

Write your final decision brief here.

*Write your answer here.*

## 7. Export and reproduce
Save `store_summary.csv`, `category_summary.csv`, `reconciliation.csv`, `sales_review.csv` and `integration_review.csv` into `case_study_outputs/`. Save the three figures as PNG files. Use `index=False` for tables.

Before submitting:
- restart the kernel and run all cells from top to bottom;
- check that joins did not multiply rows and revenue totals reconcile;
- check that raw files remain unchanged;
- remove abandoned trials and explain all final decisions;
- make sure every member can explain one cleaning decision, one join control and every chart.

The deliverable is the entire completed notebook, including interpretations, not just the first figure.

In [ ]:
# Export the evidence
# Write your code here.

### OPTIONAL / IF TIME ALLOWS — Investigate product mix
Create a store × category table of revenue or order counts. Use a labelled heatmap or compare category shares within each store. State the denominator explicitly. Does the mix help explain different order amounts? Treat this as descriptive evidence, not a causal proof.